# IE 643 - Experiment E3

**Question:** Can we get activations out, and how much disk does that cost?

**Config:** identical to E1/E2 - AnimateDiff v1.5 + `mm_sd_v15_v2`, SD1.5, 16 frames,
512x512, 25 DDIM steps, fp16, CFG 7.5, seed 42. diffusers **pinned to 0.40.0**.

**What it measures**
1. Every hookable coordinate: `motion_modules` (temporal path) and `attn1`/`attn2`
   (spatial path), indexed by `(module path, timestep)` per roadmap S5.1.
2. **Hook overhead**, against a no-hook baseline generated *in the same session on the
   same version* - the first E3 run compared against E1 across diffusers 0.37.1 vs
   0.40.0, which made the number meaningless.
3. **Full dump** bytes per clip at fp16 - the plan E3 kills.
4. **Summary store** bytes per clip, with the attention grid size as a tunable.
5. On-disk validation, and extrapolation to the 360-clip set vs Kaggle's limits.

### Kaggle setup
Accelerator -> `GPU T4 x2`, Internet -> **On**. Runtime ~5 min (two generations).
Self-contained: reuses `pipe` if present, builds it otherwise.


In [ ]:
# ============ E3 : activation extraction + storage cost ============
import sys, subprocess
from importlib.metadata import version as _pkg_version, PackageNotFoundError

# E1/E2 ran on diffusers 0.40.0; Kaggle ships an older build. Pin explicitly.
PIN = "0.40.0"

def _installed(pkg):
    try:
        return _pkg_version(pkg)
    except PackageNotFoundError:
        return None

if _installed("diffusers") != PIN:
    print(f"installing diffusers=={PIN} (found {_installed('diffusers')}) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                    f"diffusers=={PIN}"], check=True)

# If an older diffusers is already imported, drop the whole module tree.
# importlib.reload() is NOT enough: it re-executes only the top-level __init__
# while submodules in sys.modules stay on the old version, which surfaces as
# "cannot import name ... from diffusers.utils".
_stale = sys.modules.get("diffusers")
if _stale is not None and getattr(_stale, "__version__", None) != PIN:
    for _m in [m for m in list(sys.modules)
               if m == "diffusers" or m.startswith("diffusers.")]:
        del sys.modules[_m]
    if globals().pop("pipe", None) is not None:
        print("  dropped pipe built against the old diffusers")
    print("  purged stale diffusers modules")

import diffusers, transformers
print(f"diffusers {diffusers.__version__}  torch {__import__('torch').__version__}")
if diffusers.__version__ != PIN:
    raise RuntimeError(
        f"diffusers is still {diffusers.__version__}. Restart the kernel "
        "(Run -> Restart session) and re-run this cell.")

import torch, gc, re, json, os, time, numpy as np
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler

FRAMES, H, W, STEPS, CFG, SEED = 16, 512, 512, 25, 7.5, 42
PROMPT = "a golden retriever running on a beach, high quality, detailed"
NEGATIVE = "bad quality, worse quality, blurry"
ATT_GRID = 16          # attention maps downsampled to ATT_GRID x ATT_GRID per frame

ADAPTER_ID = "guoyww/animatediff-motion-adapter-v1-5-2"
BASE_CANDIDATES = ["stable-diffusion-v1-5/stable-diffusion-v1-5",
                   "botp/stable-diffusion-v1-5", "emilianJR/epiCRealism"]

def _vae_slicing(pipe):
    if hasattr(pipe.vae, "enable_slicing"):
        pipe.vae.enable_slicing()
    elif hasattr(pipe, "enable_vae_slicing"):
        pipe.enable_vae_slicing()

def build_pipe():
    adapter = MotionAdapter.from_pretrained(ADAPTER_ID, torch_dtype=torch.float16)
    last = None
    for base in BASE_CANDIDATES:
        try:
            pipe = AnimateDiffPipeline.from_pretrained(
                base, motion_adapter=adapter, torch_dtype=torch.float16)
            break
        except Exception as e:
            last = e; print(f"  [skip] {base}: {type(e).__name__}")
    else:
        raise RuntimeError(f"No SD1.5 mirror loaded. Last error: {last}")
    pipe.scheduler = DDIMScheduler.from_pretrained(
        base, subfolder="scheduler", clip_sample=False,
        timestep_spacing="linspace", beta_schedule="linear", steps_offset=1)
    pipe.set_progress_bar_config(disable=True)
    pipe.to("cuda"); _vae_slicing(pipe)
    return pipe, base

if "pipe" in globals():
    print("reusing pipe already in memory")
else:
    pipe, BASE_ID = build_pipe(); print(f"loaded {BASE_ID}")

# ---- classify every hookable coordinate -------------------------------------
# Temporal path  : the motion_modules themselves (roadmap S5.1)
# Spatial path   : transformer attn1 (self) / attn2 (cross to text)
# Attention inside a motion module is temporal, not spatial - keep them apart.
TARGETS = []
for name, mod in pipe.unet.named_modules():
    inside_motion = ".motion_modules." in name or name.endswith("motion_modules")
    if re.search(r"motion_modules\.\d+$", name):
        kind = "temporal_module"
    elif name.endswith(".attn1"):
        kind = "temporal_attn1" if inside_motion else "spatial_attn1"
    elif name.endswith(".attn2"):
        kind = "temporal_attn2" if inside_motion else "spatial_attn2"
    else:
        continue
    TARGETS.append((name, mod, kind))

from collections import Counter
print(f"\n{len(TARGETS)} hookable coordinates")
for k, n in sorted(Counter(k for _, _, k in TARGETS).items()):
    print(f"  {k:<18} {n:>4}")


In [ ]:
# ---- baseline: same session, same version, NO hooks -------------------------
def make_latents():
    g = torch.Generator("cpu").manual_seed(SEED)
    shape = (1, pipe.unet.config.in_channels, FRAMES,
             H // pipe.vae_scale_factor, W // pipe.vae_scale_factor)
    return torch.randn(shape, generator=g, dtype=torch.float16).to("cuda")

def generate():
    gc.collect(); torch.cuda.empty_cache()
    torch.cuda.synchronize(); t0 = time.perf_counter()
    out = pipe(prompt=PROMPT, negative_prompt=NEGATIVE, num_frames=FRAMES,
               height=H, width=W, num_inference_steps=STEPS,
               guidance_scale=CFG, latents=make_latents())
    torch.cuda.synchronize()
    return out, time.perf_counter() - t0

# E1 showed no measurable warm-up penalty (cold 93.6 s vs warm 94.1 s),
# so a single baseline run is enough.
_, BASE_S = generate()
print(f"baseline, no hooks   {BASE_S:6.1f} s")

# ---- hook, run again, record metadata only (never keep the tensors) ---------
records, handles = [], []
STEP = {"i": -1}
CAPTURE = {"name": None, "tensor": None}      # one real tensor, for disk validation

def _first_tensor(x):
    if torch.is_tensor(x): return x
    if isinstance(x, (tuple, list)):
        for e in x:
            t = _first_tensor(e)
            if t is not None: return t
    return None

def make_hook(name, kind):
    def hook(mod, inp, out):
        t = _first_tensor(out)
        if t is None: return
        records.append({"name": name, "kind": kind, "step": STEP["i"],
                        "shape": tuple(t.shape), "numel": t.numel(),
                        "bytes": t.numel() * 2})          # fp16
        if CAPTURE["tensor"] is None and kind == "temporal_module":
            CAPTURE["name"] = name
            CAPTURE["tensor"] = t.detach().to(torch.float16).cpu().clone()
    return hook

def step_hook(mod, inp):
    STEP["i"] += 1                                        # one UNet call per step

try:
    handles.append(pipe.unet.register_forward_pre_hook(step_hook))
    for name, mod, kind in TARGETS:
        handles.append(mod.register_forward_hook(make_hook(name, kind)))
    _, HOOK_S = generate()
finally:
    for h in handles:
        h.remove()

steps_seen = STEP["i"] + 1
OVERHEAD = 100 * (HOOK_S - BASE_S) / BASE_S
print(f"hooked ({len(TARGETS)} coords) {HOOK_S:6.1f} s")
print(f"hook overhead        {OVERHEAD:+6.1f} %   <- same session, same version")
print(f"\nUNet calls {steps_seen} for {STEPS} scheduler steps")
print(f"records captured {len(records):,}")
print(f"example record: {records[0]}")


In [ ]:
# ---- storage accounting -----------------------------------------------------
GB, MB = 1024 ** 3, 1024 ** 2
full_by_kind = {}
for r in records:
    full_by_kind[r["kind"]] = full_by_kind.get(r["kind"], 0) + r["bytes"]
FULL = sum(full_by_kind.values())

print(f"{'kind':<20}{'coords':>8}{'GB/clip':>10}{'share':>8}")
print("-" * 46)
for k, b in sorted(full_by_kind.items(), key=lambda kv: -kv[1]):
    n = len({r["name"] for r in records if r["kind"] == k})
    print(f"{k:<20}{n:>8}{b/GB:>10.3f}{100*b/FULL:>7.0f}%")
print("-" * 46)
print(f"{'FULL DUMP':<20}{len(TARGETS):>8}{FULL/GB:>10.3f}{100:>7.0f}%")

# ---- what S5.1 actually stores ---------------------------------------------
# per (module, step): channel mean + var, plus one frame-mean-removed norm per frame.
# per attention coordinate: the map downsampled to a fixed grid, per frame.
def channels(shape):
    return shape[-1] if len(shape) == 3 else (shape[1] if len(shape) >= 2 else 1)

def summary_bytes(grid):
    stats = att = 0
    for r in records:
        stats += (2 * channels(r["shape"]) + FRAMES) * 4      # fp32
        if r["kind"].endswith(("attn1", "attn2")):
            att += grid * grid * FRAMES * 4
    return stats, att

stats_b, att_b = summary_bytes(ATT_GRID)
SUMMARY = stats_b + att_b
print(f"\nsummary store at {ATT_GRID}x{ATT_GRID}: {SUMMARY/MB:.1f} MB/clip")
print(f"  channel stats      {stats_b/MB:>7.1f} MB  {100*stats_b/SUMMARY:>3.0f}%")
print(f"  attention movies   {att_b/MB:>7.1f} MB  {100*att_b/SUMMARY:>3.0f}%")
print(f"  compression vs full dump: {FULL/SUMMARY:,.0f}x")

N_CLIPS = 360                                   # slide 13: three seeds, 360 clips
OUT_CAP, DISK_CAP = 20, 73                      # Kaggle GB
print(f"\nattention grid sensitivity, over {N_CLIPS} clips:")
print(f"  {'grid':>6}{'MB/clip':>10}{'GB total':>11}{'vs 20 GB cap':>14}")
for grid in (32, 16, 8, 4):
    s, a = summary_bytes(grid)
    tot = N_CLIPS * (s + a) / GB
    print(f"  {grid:>4}x{grid:<2}{(s+a)/MB:>9.1f}{tot:>11.1f}"
          f"{100*tot/OUT_CAP:>12.0f}%  {'ok' if tot < OUT_CAP else 'OVER'}")

# ---- validate the analytic number against a real file -----------------------
t = CAPTURE["tensor"]
np.save("/kaggle/working/_e3_probe.npy", t.numpy())
on_disk, analytic = os.path.getsize("/kaggle/working/_e3_probe.npy"), t.numel() * 2
print(f"\ndisk check on {CAPTURE['name']}  shape {tuple(t.shape)}")
print(f"  analytic {analytic/MB:.2f} MB   on-disk {on_disk/MB:.2f} MB   "
      f"overhead {100*(on_disk-analytic)/analytic:+.2f}%")
os.remove("/kaggle/working/_e3_probe.npy")

print(f"\nfull dump, ONE clip   {FULL/GB:>9.1f} GB  vs {DISK_CAP} GB working disk"
      f"  -> {'fits' if FULL/GB < DISK_CAP else 'DOES NOT FIT'}")
print(f"full dump, {N_CLIPS} clips {N_CLIPS*FULL/GB:>9,.0f} GB")


In [ ]:
# ---- record the result ------------------------------------------------------
import datetime
gpu = torch.cuda.get_device_properties(0)
record = {
    "experiment": "E3", "date": datetime.date.today().isoformat(),
    "gpu": gpu.name, "diffusers": diffusers.__version__, "torch": torch.__version__,
    "frames": FRAMES, "resolution": f"{H}x{W}", "steps": STEPS,
    "guidance_scale": CFG, "seed": SEED, "unet_calls": steps_seen,
    "baseline_s_no_hooks": round(BASE_S, 1),
    "generation_s_with_hooks": round(HOOK_S, 1),
    "hook_overhead_pct": round(OVERHEAD, 1),
    "n_coordinates": len(TARGETS),
    "coordinates_by_kind": {k: len({r["name"] for r in records if r["kind"] == k})
                            for k in full_by_kind},
    "full_dump_gb_per_clip": round(FULL / GB, 3),
    "full_dump_gb_by_kind": {k: round(v / GB, 3) for k, v in full_by_kind.items()},
    "attention_grid": ATT_GRID,
    "summary_store_mb_per_clip": round(SUMMARY / MB, 2),
    "summary_split_mb": {"channel_stats": round(stats_b / MB, 2),
                         "attention_movies": round(att_b / MB, 2)},
    "compression_ratio": round(FULL / SUMMARY, 1),
    "n_clips_planned": N_CLIPS,
    "full_dump_gb_total": round(N_CLIPS * FULL / GB, 1),
    "summary_store_gb_total": round(N_CLIPS * SUMMARY / GB, 2),
}
with open("/kaggle/working/E3_result.json", "w") as f:
    json.dump(record, f, indent=2)
print(json.dumps(record, indent=2))

print("\n" + "=" * 68)
print("PASTE INTO ROADMAP S2.2 / SLIDE 10, E3 OUTCOME:")
print("=" * 68)
print(f"Easy to hook: {len(TARGETS)} coordinates, {OVERHEAD:+.0f}% generation overhead. "
      f"A full dump is {FULL/GB:.0f} GB per clip at fp16, so the harness stores "
      f"{SUMMARY/MB:.0f} MB of summaries instead ({FULL/SUMMARY:,.0f}x smaller).")
print("=" * 68)
